# Aligning full coronal sections of adult mouse brain from MERFISH and Xenium (squidpy + moscot)

In this notebook, we align two single cell resolution spatial transcriptomics datasets of full coronal sections of the adult mouse brain from approximately the same locations assayed by MERFISH and Xenium.

The original notebook used STalign LDDMM. Here we use `squidpy` with `moscot` optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load data

In [ ]:
# Source: MERFISH
fname = '../../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)

coords_source = np.column_stack([df1['center_x'].values, df1['center_y'].values])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)), obs=df1)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2)
ax.set_title('MERFISH (source)')

In [ ]:
# Target: Xenium
fname = '../../xenium_data/Xenium_V1_FF_Mouse_Brain_MultiSection_1_cells.csv.gz'
df2 = pd.read_csv(fname)

coords_target = np.column_stack([df2['x_centroid'].values, df2['y_centroid'].values])
adata_target = ad.AnnData(X=np.zeros((len(coords_target), 1)), obs=df2)
adata_target.obsm['spatial'] = coords_target

fig, ax = plt.subplots()
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, c='#ff7f0e')
ax.set_title('Xenium (target)')

In [ ]:
# Overlay
fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='MERFISH')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='Xenium')
ax.legend(markerscale=10)

## Align using moscot

In [ ]:
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='MERFISH')
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='MERFISH aligned')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='Xenium')
ax.legend(markerscale=10)